In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.gold.dim_clientes AS
SELECT customer_id, name, email, phone, state, created_at
FROM ecommerce.silver.customers;

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.gold.dim_produtos AS
SELECT product_id, product_name, category, price, is_active
FROM ecommerce.silver.products;

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.gold.fato_pedidos AS
SELECT
  o.order_id,
  o.customer_id,
  o.order_date,
  o.order_day,
  o.status,
  o.payment_method,
  o.installments,
  o.qtd_itens,
  ROUND(COALESCE(i.valor_total, 0), 2) AS valor_total,
  c.customer_id IS NULL                AS cliente_nao_cadastrado
FROM ecommerce.silver.orders o
LEFT JOIN (
  SELECT order_id, SUM(total_item) AS valor_total
  FROM ecommerce.silver.order_items
  GROUP BY order_id
) i ON i.order_id = o.order_id
LEFT JOIN ecommerce.silver.customers c ON c.customer_id = o.customer_id;

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.gold.fato_itens AS
SELECT
  i.order_id,
  o.order_day,
  o.status,
  i.product_id,
  i.quantity,
  i.unit_price,
  i.total_item,
  p.product_id IS NULL AS produto_nao_cadastrado
FROM ecommerce.silver.order_items i
JOIN ecommerce.silver.orders o ON o.order_id = i.order_id
LEFT JOIN ecommerce.silver.products p ON p.product_id = i.product_id;

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.gold.fato_eventos AS
SELECT event_id, session_id, customer_id, event_type, product_id, platform, event_timestamp
FROM ecommerce.silver.events;

In [0]:
%sql
-- VENDAS DIÁRIAS
CREATE OR REPLACE TABLE ecommerce.gold.vendas_diarias AS
SELECT
  o.order_day,
  COUNT(*)                                             AS qtd_pedidos,
  COUNT_IF(o.status = 'completed')                     AS pedidos_concluidos,
  COUNT_IF(o.status = 'canceled')                      AS pedidos_cancelados,
  ROUND(COUNT_IF(o.status = 'canceled') / COUNT(*), 4) AS taxa_cancelamento,
  ROUND(SUM(CASE WHEN o.status = 'completed' THEN i.total_item END), 2) AS receita,
  ROUND(SUM(CASE WHEN o.status = 'completed' THEN i.total_item END)
        / NULLIF(COUNT_IF(o.status = 'completed'), 0), 2) AS ticket_medio
FROM ecommerce.silver.orders o
LEFT JOIN (
  SELECT order_id, SUM(total_item) AS total_item
  FROM ecommerce.silver.order_items
  GROUP BY order_id
) i ON i.order_id = o.order_id
GROUP BY o.order_day;

In [0]:
%sql
SELECT 'fato_pedidos' AS objeto, COUNT(*) AS linhas FROM ecommerce.gold.fato_pedidos
UNION ALL SELECT 'fato_itens',      COUNT(*) FROM ecommerce.gold.fato_itens
UNION ALL SELECT 'dim_clientes',    COUNT(*) FROM ecommerce.gold.dim_clientes
UNION ALL SELECT 'dim_produtos',    COUNT(*) FROM ecommerce.gold.dim_produtos
UNION ALL SELECT 'vendas_diarias',  COUNT(*) FROM ecommerce.gold.vendas_diarias;